## Running a AFL pipleine using REST API

- This notebook requires that you have active AFL server started using the AFL-Andon app

- Before you start using the loading system, make sure nitrogen air inlet is open for at least 45 mins and about >50 psi

In [1]:
from AFL.automation.APIServer.Client import Client

In [3]:
client_ip_dict = {
    'prep':'localhost:5005',
    'load':'piloader:5000',
    # 'agent':'localhost:5053',
    'turbidity':'localhost:5001',
}
client= {}
for name,uri in client_ip_dict.items():
    ip=uri.split(':')[0]
    port=int(uri.split(':')[1])
    print(f"Connecting to AFL client {name} at {ip} : {port}")
    client[name] = Client(ip=ip,port=port)    
    client[name].login('nb')

Connecting to AFL client prep at localhost : 5005
Connecting to AFL client load at piloader : 5000
Connecting to AFL client turbidity at localhost : 5001


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: /login (Caused by NewConnectionError("HTTPConnection(host='localhost', port=5001): Failed to establish a new connection: [Errno 111] Connection refused"))

### Home the Opentrons gantry

In [50]:
client['prep'].enqueue(task_name='home')

'QD-33a58484-01aa-423e-93de-b94e415397ab'

In [52]:
# Reset stocks (clear all configured stocks)
client['prep'].enqueue(task_name='reset_stocks', interactive=True)

# Reset the deck (clear all loaded labware/instruments)
client['prep'].enqueue(task_name='reset_deck', interactive=True)

# Full reset (clears protocol session + deck)
client['prep'].enqueue(task_name='reset', interactive=True)

{'ended': '06/03/26 14:39:27-805097 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 14:39:27-714164',
 'return_val': None,
 'run_time_minutes': 0.0,
 'run_time_seconds': 0,
 'started': '06/03/26 14:39:27-803229 '}

### Load required labware

In [53]:
# Load tipracks
client['prep'].enqueue(task_name='load_labware', name='opentrons_96_tiprack_20ul', slot='5', interactive=True)
client['prep'].enqueue(task_name='load_labware', name='opentrons_96_tiprack_300ul', slot='6', interactive=True)

# Load pipettes
client['prep'].enqueue(
    task_name='load_instrument',
    name='p20_single_gen2',
    mount='left',
    tip_rack_slots=['5'],
    interactive=True
)

client['prep'].enqueue(
    task_name='load_instrument',
    name='p300_single',
    mount='right',
    tip_rack_slots=['6'],
    interactive=True
)

# Load labware
client['prep'].enqueue(task_name='load_labware', name='corning_96_wellplate_360ul_flat', slot='1', interactive=True)
client['prep'].enqueue(task_name='load_labware', name='nist_6_20ml_vials', slot='2', interactive=True)
client['prep'].enqueue(task_name='load_labware', name='nist_pneumatic_loader', slot='10', interactive=True)

{'ended': '06/03/26 14:41:49-121844 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 14:41:48-843613',
 'return_val': 'f8a341c1-fb79-40f8-a235-1b8879678425',
 'run_time_minutes': 0.0,
 'run_time_seconds': 0,
 'started': '06/03/26 14:41:48-924788 '}

### Verify that all the labware were loaded

In [54]:
client['prep'].driver_status()

['AFL Server Stocks: []',
 'Stocks: 3 configured',
 'Stock locations: {}',
 '0 preparation targets available',
 'No prep targets loaded',
 '96/96 tips available on left mount\n96/96 tips available on right mount',
 'Pipette on left mount: p20_single_v2.2',
 'Pipette on right mount: p300_single_v1.4',
 'Labware in slot 5: opentrons_96_tiprack_20ul',
 'Labware in slot 6: opentrons_96_tiprack_300ul',
 'Labware in slot 1: corning_96_wellplate_360ul_flat',
 'Labware in slot 2: nist_6_20ml_vials',
 'Labware in slot 10: nist_pneumatic_loader']

### Opentrons pipeppting and sample preparation

In [55]:
# Add components
client['prep'].enqueue(
    task_name='add_component',
    name='Red',
    formula='H2O',
    density='1.0 g/ml',
    interactive=True
)

client['prep'].enqueue(
    task_name='add_component',
    name='Blue',
    formula='H2O',
    density='1.0 g/ml',
    interactive=True
)

client['prep'].enqueue(
    task_name='add_component',
    name='Green',
    formula='H2O',
    density='1.0 g/ml',
    interactive=True
)

# Add stocks
client['prep'].enqueue(
    task_name='add_stock',
    solution={
        "name": "stock_Red",
        "location": "2A1",
        "concentrations": {"Red": "1 mg/ml"},
        "volumes": {"H2O": "20 ml"},
        "total_volume": "20 ml",
        "solutes": ["Red"],
    },
    interactive=True
)

client['prep'].enqueue(
    task_name='add_stock',
    solution={
        "name": "stock_Blue",
        "location": "2A2",
        "concentrations": {"Blue": "1 mg/ml"},
        "volumes": {"H2O": "20 ml"},
        "total_volume": "20 ml",
        "solutes": ["Blue"],
    },
    interactive=True
)

client['prep'].enqueue(
    task_name='add_stock',
    solution={
        "name": "stock_Green",
        "location": "2A3",
        "concentrations": {"Green": "1 mg/ml"},
        "volumes": {"H2O": "20 ml"},
        "total_volume": "20 ml",
        "solutes": ["Green"],
    },
    interactive=True
)

{'ended': '06/03/26 14:43:13-891972 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 14:43:13-869006',
 'return_val': None,
 'run_time_minutes': 0.0,
 'run_time_seconds': 0,
 'started': '06/03/26 14:43:13-869569 '}

In [56]:
target = {
    "name": "color_sample",
    "location": "1A1",               # optional if you pass dest explicitly
    "volume_fractions": {
        "stock_Red": 0.1,
        "stock_Blue": 0.7,
        "stock_Green": 0.2,
    },
    "total_volume": "300 ul",
}

In [57]:
# we should be able to automate this in the prep driver level
for name, frac in target["volume_fractions"].items():
    source = None
    volume = 300*frac
    for s in stocks:
        if s.name == name:
            source = s.location
    if source is None:  
        raise RuntimeError(f'Stock with name {name} not found')
    else:
        print(f"Pipetting stock {name} for a volume of {volume} to {target['location']}")
    client['prep'].enqueue(
        task_name='transfer', 
        source=source, 
        dest=target["location"], 
        volume=volume,
        interactive=True
    )


Pipetting stock stock_Red for a volume of 30.0 to 1A1
Pipetting stock stock_Blue for a volume of 210.0 to 1A1
Pipetting stock stock_Green for a volume of 60.0 to 1A1


In [58]:
print("Transfering the target solution to AFL catch using Pneumatic loader client")
client['prep'].enqueue(
    task_name = 'transfer_to_catch',
    source = target["location"],
    dest = "10A1", # not necessary as we have already loaded the module
    interactive = True
)

Transfering the target solution to AFL catch using Pneumatic loader client


{'ended': '06/03/26 14:48:04-195615 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 14:46:50-546553',
 'return_val': None,
 'run_time_minutes': 1.2166666666666666,
 'run_time_seconds': 73,
 'started': '06/03/26 14:46:50-547110 '}

### Turbidity Mesurments
1. Use turbidity cell to take an image
2. Load sample from the catch to turbdiity cell, measure, rinse and reset it for next measurement

In [59]:
turbidity_config = {
        'name':'OpticalTurbidity',
        'client_name':'inst',
        'load_dest_label':'afterTurb',
        'sample_dim':'turb_sample',
        'sample_comps_variable':'turb_comps',
        'empty_base_kw': {'task_name': 'measure','set_empty':True,'plotting':False},
        'measure_base_kw': {'task_name': 'measure','set_empty':False,'plotting':True},
        'concat_dim':'turb_sample'                  
    }
client['turbidity'].set_config(interactive=True, **turbidity_config)

{'ended': '06/03/26 14:48:41-152482 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 14:48:41-151631',
 'return_val': None,
 'run_time_minutes': 0.0,
 'run_time_seconds': 0,
 'started': '06/03/26 14:48:41-152212 '}

In [ ]:
client['load'].enqueue(task_name='loadSample',load_dest_label='afterTurb')

'QD-6d5b4842-6382-40a1-8024-194dc5caadac'

This should turn the lights green (one solid on and then off), and three flickers on the light above of the bubble sensor on the turbidity cell


In [ ]:
client['load'].enqueue(task_name='calibrate_sensor')

'QD-07b8ed86-5b72-43f9-99a8-d920056f20ac'

Need more clarification on the following

In [ ]:
import uuid
uid = 'SAM-' +  str(uuid.uuid4())
uid
for client_name,client_obj in client.items():
    client_obj.enqueue(task_name='set_sample',sample_name='test',sample_uuid=uid)

In [73]:
client['turbidity'].enqueue(task_name='measure', set_empty=True, plot=True, interactive=True)

{'ended': '06/03/26 15:10:25-380438 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 15:10:23-584678',
 'return_val': 'xarray.Dataset',
 'run_time_minutes': 0.016666666666666666,
 'run_time_seconds': 1,
 'started': '06/03/26 15:10:23-585616 '}

In [1]:
from tiled.client import from_uri
from tiled.queries import Eq,Contains, NotIn, Regex

tiled_client = from_uri(
    'http://localhost:8000/', 
    api_key='836b96fc314600105be5dd642d87065c28f47fb479933603571c6ef49de80ee6'
)

In [4]:
client_rd = tiled_client['run_documents']
client_rd.search(Eq('task_name','capture_rgb'))

<Container {}>

### Tiled:
1. This seems to keep the data from previous runs. How do I reset so that I can only look at data from the current campaign? Or should we simply filter it on the database GUI?
2. How do we locally load and analyze the data collected from the isntrument above?

In [ ]:
res = tiled_client.search(Eq('array_name','USAXS_I'))
for uid in res:
    if 'sample_uuid' in tiled_client[uid].metadata:
        print(tiled_client[uid].metadata['array_name'],tiled_client[uid].metadata['sample_uuid'],)

## Set up AFL-agent
1. Load a prefab and attach it to the agent client for analysis and recommendations
2. Define a full python driver customaizable from AFL-agent codebase classes and attach that config to the existing agent client.

In [5]:
from AFL.double_agent import *

In [6]:
list_prefabs()


Available Prefabricated Pipelines:
|-----------------------|-------------------------------------------------------------------------------------------------------------|
| Name                  | Description                                                                                                 |
|-----------------------|-------------------------------------------------------------------------------------------------------------|
| find_boundaries       | A simlarity-clustering-classification pipeline for finding boundaries in measurement data                   |
| preprocess            | A pipeline that generates a Cartesian grid, normalizes data, and calculates derivatives using Savgol filter |
| similarity_clustering | A simlarity-clustering pipeline for clustering measurements into groups                                     |
|-----------------------|-------------------------------------------------------------------------------------------------------------|
Total: 3 pre

In [9]:
load_prefab("find_boundaries").print_code()

Pipeline code has been prepared in a new cell below.


In [ ]:
with Pipeline(name = "find_boundaries") as p:
    Standardize(
        input_variable="composition",
        output_variable="normalized_composition",
        dim="sample",
        component_dim="component",
        scale_variable=None,
        min_val={'A': 0.0, 'B': 0.0},
        max_val={'A': 10.0, 'B': 25.0},
        name="Standardize",
    )

    Standardize(
        input_variable="composition_grid",
        output_variable="normalized_composition_grid",
        dim="grid",
        component_dim="component",
        scale_variable=None,
        min_val={'A': 0.0, 'B': 0.0},
        max_val={'A': 10.0, 'B': 25.0},
        name="Standardize",
    )

    SavgolFilter(
        input_variable="measurement",
        output_variable="derivative",
        dim="x",
        xlo=None,
        xhi=None,
        xlo_isel=None,
        xhi_isel=None,
        pedestal=None,
        npts=250,
        derivative=1,
        window_length=31,
        polyorder=2,
        apply_log_scale=True,
        name="SavgolFilter",
    )

    Similarity(
        input_variable="derivative",
        output_variable="similarity",
        sample_dim="sample",
        params={'metric': 'laplacian', 'gamma': 0.0001},
        constrain_same=[],
        constrain_different=[],
        name="SimilarityMetric",
    )

    SpectralClustering(
        input_variable="similarity",
        output_variable="labels",
        dim="sample",
        params={'n_phases': 2},
        name="SpectralClustering",
        use_silhouette=False,
    )

    GaussianProcessClassifier(
        feature_input_variable="normalized_composition",
        predictor_input_variable="labels",
        output_prefix="extrap",
        grid_variable="normalized_composition_grid",
        grid_dim="grid",
        sample_dim="sample",
        kernel="Matern",
        kernel_kwargs={'length_scale': 1.0, 'nu': 1.5},
        optimizer="fmin_l_bfgs_b",
        name="GaussianProcessClassifier",
    )

    MaxValueAF(
        input_variables=['extrap_entropy'],
        grid_variable="composition_grid",
        grid_dim="grid",
        combine_coeffs=None,
        output_prefix=None,
        output_variable="next_sample",
        decision_rtol=0.05,
        random_fraction=0.0,
        excluded_comps_variables=None,
        excluded_comps_dim=None,
        exclusion_radius=0.001,
        count=1,
        name="MaxValueAF",
    )



In [20]:
client['agent'].enqueue(task_name="initialize_pipeline", pipeline = p.to_dict()['ops'])

'QD-ce5b6380-b238-432b-af1a-00a59fb10da7'

In [25]:
client['agent'].get_config('tiled_input_groups',interactive=True)

{'ended': '06/03/26 15:54:37-210313 ',
 'exit_state': 'Success!',
 'queued': '06/03/26 15:54:37-208721',
 'return_val': [{'concat_dim': 'saxs',
   'entry_ids': ['QD-4ff1b8f0-4636-4667-898c-0434aca9e685',
    'QD-5c787724-97b3-4ce1-8971-9fdf77e43e00',
    'QD-ad0fed45-9ee1-4d39-903b-e1d1e5d47fd9',
    'QD-b665f140-9b5a-4fb3-baec-8d1fa6ad3826'],
   'variable_prefix': 'saxs_'}],
 'run_time_minutes': 0.0,
 'run_time_seconds': 0,
 'started': '06/03/26 15:54:37-209916 '}

In [ ]:
for i in range(budget):
    

In [ ]:
client['agent'].set_config(tiled_input_groups = )

In [ ]:
rinse_program = [  
    ['rinse1', 20], 
    [None, 2], 
    ['rinse2', 20], 
    [None, 2],
    ['blow', 20],
    [None, 2],
    ['blow', 20],
    [None, 2]
]
client['load'].set_config(rinse_program=rinse_program)

'QD-686f6fc5-8ef7-4b15-b034-01b523106afa'

In [ ]:
rinse_program = [  
    ['blow', 20],
    [None, 2],
    ['blow', 5],
    [None, 2]
]
client['load'].set_config(rinse_program=rinse_program)

'QD-ae5656ba-1cf5-4b91-88c7-ebcef53cf4c7'

In [30]:
client['load'].enqueue(task_name='loadSample',load_dest_label='afterTurb')

'QD-c5b24c6d-7bb4-4bc1-b639-79789ec04d87'

In [31]:
client['load'].enqueue(task_name='rinseCell')

'QD-c8606695-8b53-4175-8dbe-0755157edf40'